In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "8"

In [2]:
from mmdet.apis import init_detector, inference_detector
print("mmdet OK")


mmdet OK


In [3]:
import os
import cv2
import math
import numpy as np
from tqdm import tqdm

from mmpose.apis import MMPoseInferencer


In [4]:
IMG_DIR = "../../dataset/input_images"
LABEL_DIR = "../../dataset/labels"
VIS_DIR = "../outputs/keypoints_vis"

os.makedirs(LABEL_DIR, exist_ok=True)
os.makedirs(VIS_DIR, exist_ok=True)


In [5]:
inferencer = MMPoseInferencer(
    pose2d="../models/rtmpose-l_8xb32-270e_coco-wholebody-384x288.py",
    pose2d_weights="../models/rtmpose-l_simcc-coco-wholebody_pt-aic-coco_270e-384x288-eaeb96c8_20230125.pt",
    det_model="../checkpoints_det/rtmdet_l_8xb32-300e_coco.py",
    det_weights="../checkpoints_det/rtmdet_l_8xb32-300e_coco_20220719_112030-5a0be7c4.pth",
    device="cuda"
)


Loads checkpoint by local backend from path: ../models/rtmpose-l_simcc-coco-wholebody_pt-aic-coco_270e-384x288-eaeb96c8_20230125.pt
01/20 15:08:53 - mmengine - WARNING - Failed to search registry with scope "mmpose" in the "function" registry tree. As a workaround, the current "function" registry in "mmengine" is used to build instance. This may cause unexpected failure when running the built modules. Please check whether "mmpose" is a correct scope, or whether the registry is initialized.
Loads checkpoint by local backend from path: ../checkpoints_det/rtmdet_l_8xb32-300e_coco_20220719_112030-5a0be7c4.pth
The model and loaded state dict do not match exactly

unexpected key in source state_dict: data_preprocessor.mean, data_preprocessor.std

01/20 15:08:54 - mmengine - WARNING - Failed to search registry with scope "mmdet" in the "function" registry tree. As a workaround, the current "function" registry in "mmengine" is used to build instance. This may cause unexpected failure when runn

In [6]:
def compute_bbox(keypoints, img_w, img_h, margin=0.12):
    xs = [kp[0] for kp in keypoints if kp[2] > 0]
    ys = [kp[1] for kp in keypoints if kp[2] > 0]

    if len(xs) == 0:
        return None

    x_min, x_max = min(xs), max(xs)
    y_min, y_max = min(ys), max(ys)

    # margin 추가
    dx = (x_max - x_min) * margin
    dy = (y_max - y_min) * margin

    x_min = max(0, x_min - dx)
    y_min = max(0, y_min - dy)
    x_max = min(img_w, x_max + dx)
    y_max = min(img_h, y_max + dy)

    cx = ((x_min + x_max) / 2) / img_w
    cy = ((y_min + y_max) / 2) / img_h
    w = (x_max - x_min) / img_w
    h = (y_max - y_min) / img_h

    return cx, cy, w, h


In [7]:
def select_best_person(preds, score_thr=0.2, min_kpts=8):
    """
    여러 사람 중 keypoint가 가장 안정적인 사람 선택
    """
    if not preds or len(preds[0]) == 0:
        return None

    best_person = None
    best_score = 0

    for person in preds[0]:
        scores = np.array(person["keypoint_scores"])
        valid = (scores > score_thr).sum()

        if valid >= min_kpts and valid > best_score:
            best_person = person
            best_score = valid

    return best_person


In [8]:
def save_yolo_pose_label(
    path,
    bbox,          # (cx, cy, w, h) → 이미 normalize 된 값
    keypoints,     # (N, 3) : x(px), y(px), score
    img_w,
    img_h,
    score_thr=0.3
):
    """
    YOLO11 Pose annotation writer
    - bbox: normalized
    - keypoints: pixel coords → normalize here
    - v: {0,1,2} (COCO rule)
    """

    cx, cy, w, h = bbox

    with open(path, "w") as f:
        line = f"0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f} "

        for x, y, s in keypoints:
            # visibility 판단
            if s <= 0:
                v = 0
                x_n, y_n = 0.0, 0.0
            elif s < score_thr:
                v = 1
                x_n = x / img_w
                y_n = y / img_h
            else:
                v = 2
                x_n = x / img_w
                y_n = y / img_h

            line += f"{x_n:.6f} {y_n:.6f} {v} "

        f.write(line.strip())


In [9]:
KEYPOINT_MAPPING = {
    # COCO 17
    0: 0,    # nose
    1: 1,    # left_eye
    2: 2,    # right_eye
    3: 3,    # left_ear
    4: 4,    # right_ear
    5: 5,    # left_shoulder
    6: 6,    # right_shoulder
    7: 7,    # left_elbow
    8: 8,    # right_elbow
    9: 9,    # left_wrist
    10: 10,  # right_wrist
    11: 11,  # left_hip
    12: 12,  # right_hip
    13: 13,  # left_knee
    14: 14,  # right_knee
    15: 15,  # left_ankle
    16: 16,  # right_ankle

    # Foot extra
    17: 19,  # left_heel
    18: 22,  # right_heel
    19: 17,  # left_big_toe
    20: 20   # right_big_toe
}

# YOLO 기준 스켈레톤
SKELETON = [
    (5, 7), (7, 9),      # left arm
    (6, 8), (8, 10),     # right arm
    (5, 6),              # shoulders
    (11, 13), (13, 15),  # left leg
    (12, 14), (14, 16),  # right leg
    (11, 12),            # hips
    (15, 17), (17, 19),  # left foot
    (16, 18), (18, 20),  # right foot
]



In [10]:
def extract_21_keypoints(mmpose_result):
    """
    return: (21, 3) -> x, y, score
    """
    person = mmpose_result["predictions"][0][0]
    kps = person["keypoints"]          # (133, 2)
    scores = person["keypoint_scores"] # (133,)

    out = np.zeros((21, 3), dtype=np.float32)

    for new_idx, mmpose_idx in KEYPOINT_MAPPING.items():
        out[new_idx, 0] = kps[mmpose_idx][0]
        out[new_idx, 1] = kps[mmpose_idx][1]
        out[new_idx, 2] = scores[mmpose_idx]

    return out


In [11]:
def draw_keypoints(img, keypoints):
    for x, y, v in keypoints:
        if v > 0:
            cv2.circle(img, (int(x), int(y)), 4, (0, 255, 0), -1)
    return img



def draw_skeletons(img, keypoints, score_thr=0.3):
    img = img.copy()

    # draw points
    for i, (x, y, s) in enumerate(keypoints):
        if s > score_thr:
            cv2.circle(img, (int(x), int(y)), 4, (0, 255, 0), -1)
            cv2.putText(
                img, str(i),
                (int(x)+3, int(y)-3),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.4, (255, 0, 0), 1
            )

    # draw skeleton
    for a, b in SKELETON:
        if keypoints[a][2] > score_thr and keypoints[b][2] > score_thr:
            pt1 = tuple(map(int, keypoints[a][:2]))
            pt2 = tuple(map(int, keypoints[b][:2]))
            cv2.line(img, pt1, pt2, (0, 255, 255), 2)

    return img


In [12]:
def is_valid_pose(
    keypoints,
    score_thr=0.3,
    min_valid_kpts=14,
    required_ids=(11, 12, 15, 16, 19, 20)
):
    """
    keypoints: (21, 3)
    return: True / False
    """

    scores = keypoints[:, 2]

    # 1. 전체 유효 개수
    valid_cnt = (scores >= score_thr).sum()
    if valid_cnt < min_valid_kpts:
        return False

    # 2. 필수 keypoint 체크
    for idx in required_ids:
        if scores[idx] < score_thr:
            return False

    return True


In [13]:
def create_annotation(
    mmpose_result, img, label_path
):
    h, w = img.shape[:2]

    keypoints = extract_21_keypoints(mmpose_result)

    bbox = compute_bbox(keypoints, w, h)
    if bbox is None:
        return False

    save_yolo_pose_label(
        label_path, bbox, keypoints, w, h
    )

    return True


In [14]:
def process_image(
    inferencer,
    img_path,
    label_path,
    vis_path,
    score_thr=0.2
):
    """
    inferencer : MMPoseInferencer
    img_path   : 입력 이미지 경로
    label_path : YOLO annotation txt 저장 경로
    vis_path   : 시각화 이미지 저장 경로
    """

    # 1. 추론
    result = next(
        inferencer(
            str(img_path),
            return_vis=False,
            show=False
        )
    )

    preds = result["predictions"]
    person = select_best_person(preds, score_thr=score_thr, min_kpts=8)
    
    if person is None:
        print(f"❌ 유효한 사람 없음: {img_path.name}")
        return False

    # 2. 이미지 로드
    img = cv2.imread(str(img_path))
    h, w = img.shape[:2]

    # 3. keypoints (21개) 추출
    kps21 = extract_21_keypoints(result)

    # 품질 검증 -> 품질 나쁜 POSE 제거
    if not is_valid_pose(kps21):
        print(f"⚠️ skip (bad pose): {img_path.name}")
        return False

    # 4. bbox 계산
    bbox = compute_bbox(kps21, w, h)
    if bbox is None:
        print(f"❌ bbox 실패: {img_path.name}")
        return False

    # 5. YOLO annotation 저장
    save_yolo_pose_label(
        path=label_path,
        bbox=bbox,
        keypoints=kps21,
        img_w=w,
        img_h=h
    )

    # 6. 시각화
    vis_img = draw_skeletons(img, kps21, score_thr=score_thr)
    cv2.imwrite(str(vis_path), vis_img)
    return True


In [15]:
image_files = sorted([
    f for f in os.listdir(IMG_DIR)
    if f.lower().endswith((".jpg", ".png"))
])

batch_size = 64
batches = [
    image_files[i:i + batch_size]
    for i in range(0, len(image_files), batch_size)
]

print(f"Total images: {len(image_files)}")
print(f"Total batches: {len(batches)}")


Total images: 20507
Total batches: 321


In [16]:
from pathlib import Path
from tqdm import tqdm

success_cnt = 0
fail_cnt = 0
fail_list = []

print(f"🚀 Start batch processing")
print(f"Total images : {len(image_files)}")
print(f"Total batches: {len(batches)}")

for batch_idx, batch in enumerate(batches):
    print(f"\n📦 Batch {batch_idx + 1}/{len(batches)} 시작 "
          f"(누적 성공: {success_cnt}, 실패: {fail_cnt})")

    for fname in tqdm(batch, leave=False):
        img_path = Path(IMG_DIR) / fname
        label_path = Path(LABEL_DIR) / (img_path.stem + ".txt")
        vis_path = Path(VIS_DIR) / img_path.name

        ok = process_image(
            inferencer=inferencer,
            img_path=img_path,
            label_path=label_path,
            vis_path=vis_path,
            score_thr=0.3
        )

        if ok:
            success_cnt += 1
        else:
            fail_cnt += 1
            fail_list.append(fname)

    print(f"📦 Batch {batch_idx + 1} 완료 "
          f"(누적 성공: {success_cnt}, 실패: {fail_cnt})")

print("\n✅ Batch processing finished")
print(f"✔ Total success: {success_cnt}")
print(f"❌ Total failed : {fail_cnt}")


🚀 Start batch processing
Total images : 20507
Total batches: 321

📦 Batch 1/321 시작 (누적 성공: 0, 실패: 0)


  0%|          | 0/64 [00:00<?, ?it/s]

01/20 15:09:09 - mmengine - WARNING - Support for mmpose and mmdet versions up to 3.1.0 will be discontinued in upcoming releases. To ensure ongoing compatibility, please upgrade to mmdet version 3.2.0 or later.


/home/j-i14a203/.conda/envs/MMpose/lib/python3.8/site-packages/torch/functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3483.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
  3%|▎         | 2/64 [00:00<00:16,  3.65it/s]

⚠️ skip (bad pose): 000000000077.jpg


...

 85%|████████▌ | 23/27 [00:02<00:00,  9.09it/s]

⚠️ skip (bad pose): HICO_train2015_00036804.jpg
⚠️ skip (bad pose): HICO_train2015_00036823.jpg


 93%|█████████▎| 25/27 [00:02<00:00,  9.20it/s]

⚠️ skip (bad pose): HICO_train2015_00037286.jpg
⚠️ skip (bad pose): HICO_train2015_00037422.jpg


⚠️ skip (bad pose): HICO_train2015_00037825.jpg
⚠️ skip (bad pose): HICO_train2015_00038133.jpg
📦 Batch 321 완료 (누적 성공: 6331, 실패: 14176)

✅ Batch processing finished
✔ Total success: 6331
❌ Total failed : 14176


### 테스트

In [16]:
from pathlib import Path
import cv2

# 이미지 폴더
IMG_DIR = Path("../../dataset/images")

# 결과 저장 폴더
VIS_DIR = Path("../outputs/keypoints_vis")
VIS_DIR.mkdir(parents=True, exist_ok=True)

# 테스트용: 딱 1장만
img_paths = sorted(IMG_DIR.glob("*.jpg"))[:1]

print(f"테스트 이미지 수: {len(img_paths)}")


테스트 이미지 수: 1


In [17]:
for img_path in img_paths:
    print(f"Processing: {img_path.name}")

    result = next(
        inferencer(
            str(img_path),
            return_vis=False,
            show=False
        )
    )

    preds = result["predictions"]

    if not preds:
        print("❌ 사람 없음")
        continue

    # ✅ 단일 이미지 → preds[0] 이 첫 사람
    persons = preds[0]
    person = max(persons, key=lambda x: x["bbox_score"])

    print("keys:", person.keys())
    print("keypoints shape:", len(person["keypoints"]))


Processing: 000000000785.jpg
01/20 13:36:55 - mmengine - WARNING - Support for mmpose and mmdet versions up to 3.1.0 will be discontinued in upcoming releases. To ensure ongoing compatibility, please upgrade to mmdet version 3.2.0 or later.


/home/j-i14a203/.conda/envs/MMpose/lib/python3.8/site-packages/torch/functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3483.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


keys: dict_keys(['keypoints', 'keypoint_scores', 'bbox', 'bbox_score'])
keypoints shape: 133


In [22]:
from pathlib import Path

IMG_DIR = Path("../data/images_test")
LABEL_DIR = Path("../data/labels")
VIS_DIR = Path("../outputs/visualization")

LABEL_DIR.mkdir(parents=True, exist_ok=True)
VIS_DIR.mkdir(parents=True, exist_ok=True)

img_path = IMG_DIR / "test.jpg"

process_image(
    inferencer=inferencer,
    img_path=img_path,
    label_path=LABEL_DIR / "test.txt",
    vis_path=VIS_DIR / "test.jpg"
)

True